# CatVTON-Flux modular — E2E validation (virtual try-on)

Training-free try-on: concatenate garment+person into one canvas and inpaint the person's clothing region
(CatVTON on FLUX.1-Fill + the catvton LoRA). Steps: auto-mask (segformer) → **raw-path spike** (proves the
LoRA+Fill mechanism) → publish PRIVATE `remyxai/catvton-flux-modular` → **modular load** via `trust_remote_code`.

Runtime: A100 · `HUGGINGFACE_TOKEN` · accept **FLUX.1-dev** AND **FLUX.1-Fill-dev** licenses.

## 1 · Install

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate peft sentencepiece protobuf opencv-python

## 2 · GPU + auth

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 3 · Test pair + auto-mask (segformer clothes)
Person + garment from the catvton-flux examples; the agnostic mask is generated from the person photo.

In [ ]:
import requests, numpy as np, cv2
from PIL import Image
from io import BytesIO
W, H = 576, 768
def fetch(url): return Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
person  = fetch("https://raw.githubusercontent.com/nftblackmagic/catvton-flux/main/example/person/1.jpg").resize((W,H))
garment = fetch("https://raw.githubusercontent.com/nftblackmagic/catvton-flux/main/example/garment/00035_00.jpg").resize((W,H))

# auto agnostic-mask: segment the person's clothing region, white = replace
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
proc = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
seg  = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(DEV).eval()
with torch.no_grad():
    inp = proc(images=person, return_tensors="pt").to(DEV)
    logits = seg(**inp).logits
    up = torch.nn.functional.interpolate(logits, size=(H,W), mode="bilinear", align_corners=False)
    labels = up.argmax(1)[0].cpu().numpy()
# agnostic upper-body mask: cover the garment region + ARMS (so sleeves can form),
# exclude the lower body (keep the jeans), then close/dilate/fill to a solid region.
UPPER = {4, 7}          # upper-clothes, dress  (garment region to replace)
ARMS  = {14, 15}        # left-arm, right-arm   (skin labels -> needed so long sleeves have somewhere to go)
region = np.isin(labels, list(UPPER | ARMS)).astype(np.uint8) * 255
region = cv2.morphologyEx(region, cv2.MORPH_CLOSE, np.ones((25,25), np.uint8))
region = cv2.dilate(region, np.ones((15,15), np.uint8), iterations=2)
cnts, _ = cv2.findContours(region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
region = cv2.drawContours(np.zeros_like(region), cnts, -1, 255, thickness=-1)  # fill holes -> solid
mask = Image.fromarray(region).convert("RGB")
from IPython.display import display
prev = Image.new("RGB",(W*3+20,H),"white"); prev.paste(person,(0,0)); prev.paste(garment,(W+10,0)); prev.paste(mask,(2*W+20,0))
print("person | garment | auto-mask"); display(prev.resize((720,320)))

## 4 · Milestone A — raw-path spike (FLUX.1-Fill + CatVTON LoRA)
Proves the mechanism in current diffusers before we wrap it.

In [ ]:
import torch, gc
from torchvision import transforms
from diffusers import FluxTransformer2DModel, FluxFillPipeline
FILL, BASE = "black-forest-labs/FLUX.1-Fill-dev", "black-forest-labs/FLUX.1-dev"
tr = FluxTransformer2DModel.from_pretrained(FILL, subfolder="transformer", torch_dtype=DT)
sd, alphas = FluxFillPipeline.lora_state_dict("xiaozaa/catvton-flux-lora-alpha",
              weight_name="pytorch_lora_weights.safetensors", return_alphas=True)
FluxFillPipeline.load_lora_into_transformer(state_dict=sd, network_alphas=alphas, transformer=tr)
pipe = FluxFillPipeline.from_pretrained(BASE, transformer=tr, torch_dtype=DT).to(DEV)

to01 = transforms.ToTensor()   # [0,1]; FluxFillPipeline normalizes internally (current diffusers expects [0,1])
canvas   = torch.cat([to01(garment), to01(person)], dim=2)
ext_mask = torch.cat([torch.zeros(1,H,W), to01(mask)[:1]], dim=2)
g = torch.Generator(DEV).manual_seed(0)
res = pipe(height=H, width=W*2, image=canvas, mask_image=ext_mask, num_inference_steps=30,
           guidance_scale=30, generator=g, max_sequence_length=512,
           prompt="The pair of images highlights a clothing and its styling on a model, high resolution, 4K, 8K; [IMAGE1] Detailed product shot of a clothing [IMAGE2] The same cloth is worn by a model in a lifestyle setting.").images[0]
tryon_raw = res.crop((W,0,W*2,H)); tryon_raw.save("tryon_raw.png")
print("[MILESTONE A] raw try-on OK"); display(tryon_raw)
del pipe, tr; gc.collect(); torch.cuda.empty_cache()

## 5 · Milestone B — publish PRIVATE + load modular (trust_remote_code)
Upload block.py first (files.upload), then publish + load.

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"CatVTONFluxBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.CatVTONFluxBlock"}},indent=2))
FILL,BASE="black-forest-labs/FLUX.1-Fill-dev","black-forest-labs/FLUX.1-dev"
def c(r,s,l,cl): return [None,None,{"pretrained_model_name_or_path":r,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"CatVTONFluxBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c(BASE,"text_encoder","transformers","CLIPTextModel"),"tokenizer":c(BASE,"tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c(BASE,"text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c(BASE,"tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c(FILL,"transformer","diffusers","FluxTransformer2DModel"),"vae":c(BASE,"vae","diffusers","AutoencoderKL"),
 "scheduler":c(BASE,"scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/catvton-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published:", api.list_repo_files(REPO))

In [ ]:
import torch, gc
from diffusers import ModularPipeline
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/catvton-flux-modular", trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)   # expect CatVTONFluxBlock
pipe.load_components(dtype=DT); pipe.to(DEV)
g = torch.Generator(DEV).manual_seed(0)
tryon = pipe(person_image=person, garment_image=garment, mask=mask, height=H, width=W,
             num_inference_steps=30, guidance_scale=30, generator=g).images[0]
tryon.save("tryon_modular.png")
out = Image.new("RGB",(W*3+20,H),"white"); out.paste(person,(0,0)); out.paste(garment,(W+10,0)); out.paste(tryon,(2*W+20,0))
out.save("catvton_demo.png")
print("[MILESTONE B] modular try-on OK — person | garment | try-on"); display(out.resize((720,320)))

## Verdict
`loaded block: CatVTONFluxBlock` + a clean try-on (garment worn, identity/pose preserved) matching the raw path = the modular pipeline works end-to-end. Then: publish public + add to the collection.